# Section 02: Managing and Provisioning Cloud Infrastructure

**Companion notebook for**: `02-provisioning-infrastructure.html`

## Learning Objectives
- Configure load balancers and CDN via gcloud
- Manage Cloud Storage lifecycle policies programmatically
- Create and manage GKE clusters and Cloud Run services
- Work with Vertex AI APIs for ML workflows
- Call prebuilt AI APIs (Vision, NLP, Translation)

## Prerequisites
- A Google Cloud project with billing enabled
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-compute google-cloud-storage google-cloud-container google-cloud-vision google-cloud-language google-cloud-translate vertexai

In [ ]:
import os, json
PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'your-project-id')
REGION = 'us-central1'
print(f'Project: {PROJECT_ID}\nRegion: {REGION}')
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab.')
except ImportError:
    print('Not in Colab -- using local auth.')

---
## Section 1: Load Balancer Decision Matrix

Understand which load balancer type to use for different scenarios.

In [ ]:
lb_types = [
    {'name': 'External HTTP(S)', 'protocol': 'HTTP/HTTPS', 'scope': 'Global', 'cloud_armor': True, 'cdn': True, 'use_case': 'Web apps, APIs'},
    {'name': 'External SSL Proxy', 'protocol': 'SSL/TLS', 'scope': 'Global', 'cloud_armor': False, 'cdn': False, 'use_case': 'Non-HTTP SSL'},
    {'name': 'External TCP Proxy', 'protocol': 'TCP', 'scope': 'Global', 'cloud_armor': False, 'cdn': False, 'use_case': 'Non-HTTP TCP'},
    {'name': 'External Network', 'protocol': 'TCP/UDP', 'scope': 'Regional', 'cloud_armor': False, 'cdn': False, 'use_case': 'Gaming, UDP, IP preservation'},
    {'name': 'Internal HTTP(S)', 'protocol': 'HTTP/HTTPS', 'scope': 'Regional', 'cloud_armor': False, 'cdn': False, 'use_case': 'Internal microservices'},
    {'name': 'Internal TCP/UDP', 'protocol': 'TCP/UDP', 'scope': 'Regional', 'cloud_armor': False, 'cdn': False, 'use_case': 'Internal databases'},
]
print(f"{'Load Balancer':<25} {'Protocol':<12} {'Scope':<10} {'Armor':<7} {'CDN':<5} {'Use Case'}")
print('-' * 95)
for lb in lb_types:
    print(f"{lb['name']:<25} {lb['protocol']:<12} {lb['scope']:<10} {'Y' if lb['cloud_armor'] else 'N':<7} {'Y' if lb['cdn'] else 'N':<5} {lb['use_case']}")

---
## Section 2: Cloud Storage Management

Work with Cloud Storage lifecycle policies and configurations.

In [ ]:
from google.cloud import storage

def create_lifecycle_rules():
    """Generate lifecycle rules for cost optimization."""
    rules = [
        {'action': {'type': 'SetStorageClass', 'storageClass': 'NEARLINE'},
         'condition': {'age': 30, 'matchesStorageClass': ['STANDARD']}},
        {'action': {'type': 'SetStorageClass', 'storageClass': 'COLDLINE'},
         'condition': {'age': 90, 'matchesStorageClass': ['NEARLINE']}},
        {'action': {'type': 'SetStorageClass', 'storageClass': 'ARCHIVE'},
         'condition': {'age': 365, 'matchesStorageClass': ['COLDLINE']}},
        {'action': {'type': 'Delete'},
         'condition': {'age': 730}},
    ]
    return rules

rules = create_lifecycle_rules()
print('Cloud Storage Lifecycle Rules:')
print(json.dumps({'rule': rules}, indent=2))

# Storage class comparison
classes = [
    ('Standard', 'None', '$0.020/GB', '$0.00', 'Hot data'),
    ('Nearline', '30 days', '$0.010/GB', '$0.01/GB', 'Monthly access'),
    ('Coldline', '90 days', '$0.004/GB', '$0.02/GB', 'Quarterly access'),
    ('Archive', '365 days', '$0.0012/GB', '$0.05/GB', 'Yearly access'),
]
print(f"\n{'Class':<12} {'Min Duration':<14} {'Storage':<12} {'Retrieval':<12} {'Use Case'}")
print('-' * 65)
for c in classes:
    print(f'{c[0]:<12} {c[1]:<14} {c[2]:<12} {c[3]:<12} {c[4]}')

---
## Section 3: GKE Cluster Configuration

Compare Autopilot vs Standard modes and display creation commands.

In [ ]:
gke_comparison = {
    'Feature': ['Node Management', 'Pricing', 'GPU Support', 'DaemonSets', 'Privileged Containers', 'SSH to Nodes', 'SLA (regional)', 'Best For'],
    'Autopilot': ['Google-managed', 'Per pod', 'Yes (resource requests)', 'Limited', 'Not allowed', 'Not allowed', '99.95%', 'Zero node ops'],
    'Standard': ['You manage', 'Per node (VM)', 'Yes (GPU node pools)', 'Full support', 'Allowed', 'Allowed', '99.95%', 'Full K8s control'],
}

print(f"{'Feature':<25} {'Autopilot':<30} {'Standard'}")
print('-' * 80)
for i, feat in enumerate(gke_comparison['Feature']):
    print(f"{feat:<25} {gke_comparison['Autopilot'][i]:<30} {gke_comparison['Standard'][i]}")

print('\n## GKE Autopilot Creation Command:')
print('gcloud container clusters create-auto my-autopilot \\')
print('    --region=us-central1 --release-channel=regular \\')
print('    --enable-private-nodes --master-ipv4-cidr=172.16.0.0/28')

print('\n## GKE Standard with GPU Pool:')
print('gcloud container clusters create my-standard \\')
print('    --region=us-central1 --num-nodes=2 \\')
print('    --enable-autoscaling --min-nodes=1 --max-nodes=5 \\')
print('    --machine-type=e2-standard-4 --enable-ip-alias \\')
print('    --workload-pool=PROJECT_ID.svc.id.goog')

---
## Section 4: Cloud Run Configuration

Demonstrate Cloud Run deployment patterns.

In [ ]:
cloud_run_configs = {
    'Basic API': {
        'image': 'gcr.io/PROJECT/api:v1',
        'memory': '512Mi', 'cpu': '1', 'concurrency': 80,
        'min_instances': 0, 'max_instances': 100,
        'auth': '--allow-unauthenticated',
    },
    'Internal Service': {
        'image': 'gcr.io/PROJECT/worker:v1',
        'memory': '1Gi', 'cpu': '2', 'concurrency': 1,
        'min_instances': 1, 'max_instances': 50,
        'auth': '--no-allow-unauthenticated',
    },
    'Background Processor': {
        'image': 'gcr.io/PROJECT/processor:v1',
        'memory': '2Gi', 'cpu': '4', 'concurrency': 1,
        'min_instances': 0, 'max_instances': 20,
        'auth': '--no-allow-unauthenticated',
    },
}

for name, config in cloud_run_configs.items():
    print(f'## {name}')
    print(f'gcloud run deploy {name.lower().replace(" ", "-")} \\')
    print(f'    --image={config["image"]} \\')
    print(f'    --memory={config["memory"]} --cpu={config["cpu"]} \\')
    print(f'    --concurrency={config["concurrency"]} \\')
    print(f'    --min-instances={config["min_instances"]} --max-instances={config["max_instances"]} \\')
    print(f'    {config["auth"]} --region=us-central1')
    print()

---
## Section 5: Prebuilt AI APIs

Call Cloud Vision and Cloud Natural Language APIs.

In [ ]:
# Cloud Vision API
try:
    from google.cloud import vision
    client = vision.ImageAnnotatorClient()
    image = vision.Image()
    image.source.image_uri = 'https://storage.googleapis.com/cloud-samples-data/vision/label/wakeupcat.jpg'
    response = client.label_detection(image=image, max_results=5)
    print('Cloud Vision -- Label Detection:')
    for label in response.label_annotations:
        print(f'  {label.description:<30} {label.score:.1%}')
except Exception as e:
    print(f'Vision API failed: {e}')
    print('\nMock output:')
    for label, score in [('Cat', 0.98), ('Felidae', 0.96), ('Whiskers', 0.93), ('Carnivore', 0.89), ('Tabby cat', 0.87)]:
        print(f'  {label:<30} {score:.1%}')

In [ ]:
# Cloud Natural Language API
try:
    from google.cloud import language_v1
    client = language_v1.LanguageServiceClient()
    texts = [
        'Google Cloud is an excellent platform for building scalable applications.',
        'The service was terrible and completely unreliable.',
        'The product is okay but nothing special.',
    ]
    for text in texts:
        doc = language_v1.Document(content=text, type_=language_v1.Document.Type.PLAIN_TEXT)
        sentiment = client.analyze_sentiment(document=doc).document_sentiment
        label = 'Positive' if sentiment.score > 0.2 else 'Negative' if sentiment.score < -0.2 else 'Neutral'
        print(f'{label:<10} (score={sentiment.score:+.2f}) {text[:60]}')
except Exception as e:
    print(f'NL API failed: {e}')
    print('\nMock output:')
    print('Positive  (score=+0.80) Google Cloud is an excellent platform...')
    print('Negative  (score=-0.70) The service was terrible...')
    print('Neutral   (score=+0.10) The product is okay...')

---
## Section 6: Vertex AI Workflow Overview

In [ ]:
ml_workflow = [
    ('1. Data Preparation', 'Cloud Storage, BigQuery, Dataflow', 'Store and transform training data'),
    ('2. Feature Engineering', 'Vertex AI Feature Store', 'Centralize features, prevent skew'),
    ('3. Training', 'AutoML or Custom Training', 'Train model on Vertex AI compute'),
    ('4. Evaluation', 'Vertex AI Model Evaluation', 'Compare metrics against baselines'),
    ('5. Deployment', 'Vertex AI Endpoints', 'Online or batch prediction serving'),
    ('6. Monitoring', 'Vertex AI Model Monitoring', 'Detect drift, skew, and degradation'),
    ('7. Retraining', 'Vertex AI Pipelines', 'Automated retraining on schedule or trigger'),
]

print('Vertex AI ML Workflow')
print('=' * 80)
for step, service, desc in ml_workflow:
    print(f'{step:<25} {service:<35} {desc}')

---
## Section 7: Terraform GKE Cluster

In [ ]:
tf_gke = '''resource "google_container_cluster" "primary" {
  name     = "production-cluster"
  location = "us-central1"

  remove_default_node_pool = true
  initial_node_count       = 1

  workload_identity_config {
    workload_pool = "${var.project_id}.svc.id.goog"
  }

  private_cluster_config {
    enable_private_nodes    = true
    enable_private_endpoint = false
    master_ipv4_cidr_block  = "172.16.0.0/28"
  }
}

resource "google_container_node_pool" "primary" {
  name       = "primary-pool"
  cluster    = google_container_cluster.primary.id
  node_count = 2

  autoscaling {
    min_node_count = 1
    max_node_count = 10
  }

  node_config {
    machine_type = "e2-standard-4"
    disk_size_gb = 100
    disk_type    = "pd-ssd"
    oauth_scopes = ["https://www.googleapis.com/auth/cloud-platform"]
    workload_metadata_config { mode = "GKE_METADATA" }
  }
}
'''
print(tf_gke)

---
## Summary

This notebook covered PCA Section 02:

1. **Load Balancer Selection** -- Decision matrix for all 6 LB types
2. **Cloud Storage** -- Lifecycle policies and storage class comparison
3. **GKE** -- Autopilot vs Standard comparison, creation commands
4. **Cloud Run** -- Deployment patterns for different workload types
5. **Prebuilt AI APIs** -- Vision and NLP API calls
6. **Vertex AI Workflow** -- End-to-end ML pipeline stages
7. **Terraform GKE** -- Production cluster configuration

**Next**: Section 03 covers security and compliance.